# 容量约束车辆路径问题(CVRP)

**类别：** 路径

来源：[https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-cvrp](https://www.hexaly.com/templates/capacitated-vehicle-routing-problem-cvrp)


## 问题描述

**在容量约束车辆路径问题(Capacitated Vehicle Routing Problem, CVRP)**中,一组具有相同容量的配送车辆必须为具有单一商品已知需求的客户提供服务。车辆从一个共同的配送中心出发并返回。每个客户必须恰好由一辆车服务,且每辆车服务的客户总需求不得超过其容量。目标是最小化总行驶距离,同时最小化所使用的车辆数量。


### 学习要点

- 添加 list decision variables 以建模每辆卡车的客户序列
- 通过 `partition` 约束保证每个客户恰好由一辆卡车服务
- 定义 lambda 函数 来计算行驶距离
- 通过 `count` 算子统计使用的卡车数


## 数据

所提供的容量约束车辆路径问题(CVRP)算例来自 [Augerat 等人的 Set A 数据集](http://neo.lcc.uma.es/vrp/vrp-instances/capacitated-vrp-instances/)。它们遵循 [TSPLib 格式](http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/DOC.PS):

- 节点数量由关键字 DIMENSION 指定(其中包括一个配送中心,因此客户数量为节点数减 1)。
- 卡车容量由关键字 CAPACITY 指定。
- 边的类型由 EDGE_WEIGHT_TYPE 指定。请注意,在我们的模型中仅接受 EUC_2D 这一种边类型。
- 在关键字 NODE_COORD_SECTION 之后:每个节点的 ID 以及其 x、y 坐标。
- 在关键字 DEMAND_SECTION 之后:每个节点的 ID 及其需求。
- 配送中心列在关键字 DEPOT_SECTION 之后。请注意,在我们的模型中仅接受一个配送中心。

可用车辆数量等于客户数量。


## 建模方法

容量约束车辆路径问题(CVRP)的 OptAgent 模型使用 list decision variables。对每辆卡车,我们定义一个列表变量,表示其所访问的客户序列。通过对所有列表施加 partition 约束,我们确保每个客户恰好由一辆卡车服务。


当一辆卡车至少访问一个客户时,它才被车队所使用。借助 count 算子(返回列表中的元素数量),我们可以检查每辆卡车是否被使用,从而计算车队中使用的卡车总数。

我们可以使用需求数组上的 at 算子来访问序列中每个客户的需求。每辆卡车所配送的总量通过一个 lambda 函数计算,该函数将 `sum` 算子应用于所有被访问的客户。请注意,该 sum 中的项数以及列表的大小在搜索过程中会变化。

从一个客户到下一个客户所行驶的距离同样使用二维距离矩阵上的 at 算子来访问。我们使用另一个 lambda 函数 计算每辆卡车的总行驶距离,该函数对序列中相邻客户之间的距离求和。

## OptAgent 适配说明

Hexaly 原模型对每辆卡车使用 list 变量并通过 partition 约束保证客户唯一分配;同时它定义两个目标(车辆数、总距离),并按字典序最小化。OptAgent 当前的内核一次只接受一个标量目标(多目标被报告为 `unsupported_objective_mode`)。为保留原模型语义,我们将两个目标线性组合为

`combined = nb_trucks_used * TRUCK_PENALTY + total_distance`,

其中 `TRUCK_PENALTY` 是一个大于任何可行 `total_distance` 的常数;这样在最优解上每减少一辆卡车,目标值就会下降至少一个 `TRUCK_PENALTY`,从而优先压缩车辆数。同时,OptAgent 内核对默认空列表的 partition + 相邻下标 lambda(`sequence[i-1]`,`sequence[i]`)组合难以构造可行起点。我们为每个列表显式提供完整默认 `tuple(range(nb_customers))`,使初始构造阶段立即拥有可行指派。


## Python 实现


In [ ]:
from optagent import ModelBuilder, solve

import math
from pathlib import Path


def read_cvrp_instance(filename):
    """Read an Augerat-format CVRP instance and return a dict of arrays."""
    tokens = Path(filename).read_text().split()
    nb_nodes = 0
    truck_capacity = 0
    customers_x: list[int] = []
    customers_y: list[int] = []
    depot_x = 0
    depot_y = 0
    demands: list[int] = []
    index = 0
    while index < len(tokens):
        token = tokens[index]
        if token == "DIMENSION":
            nb_nodes = int(tokens[index + 2])
            index += 3
            continue
        if token == "CAPACITY":
            truck_capacity = int(tokens[index + 2])
            index += 3
            continue
        if token == "EDGE_WEIGHT_TYPE":
            if tokens[index + 2] != "EUC_2D":
                raise ValueError(f"Only EUC_2D is supported, got {tokens[index + 2]}")
            index += 3
            continue
        if token == "NODE_COORD_SECTION":
            index += 1
            break
        index += 1
    nb_customers = nb_nodes - 1
    customers_x = [0] * nb_customers
    customers_y = [0] * nb_customers
    for n in range(nb_nodes):
        node_id = int(tokens[index]); index += 1
        if node_id == 1:
            depot_x = int(tokens[index]); index += 1
            depot_y = int(tokens[index]); index += 1
        else:
            customers_x[node_id - 2] = int(tokens[index]); index += 1
            customers_y[node_id - 2] = int(tokens[index]); index += 1
    # DEMAND_SECTION
    while index < len(tokens) and tokens[index] != "DEMAND_SECTION":
        index += 1
    index += 1
    demands = [0] * nb_customers
    for n in range(nb_nodes):
        node_id = int(tokens[index]); index += 1
        demand = int(tokens[index]); index += 1
        if node_id != 1:
            demands[node_id - 2] = demand
    distance_matrix = [
        [compute_dist(customers_x[i], customers_x[j], customers_y[i], customers_y[j])
         for j in range(nb_customers)]
        for i in range(nb_customers)
    ]
    distance_depot = [
        compute_dist(depot_x, customers_x[i], depot_y, customers_y[i])
        for i in range(nb_customers)
    ]
    return {
        "nb_customers": nb_customers,
        "nb_trucks": nb_customers,
        "truck_capacity": truck_capacity,
        "distance_matrix": distance_matrix,
        "distance_depot": distance_depot,
        "demands": demands,
    }


def compute_dist(xi, xj, yi, yj):
    exact_dist = math.sqrt(math.pow(xi - xj, 2) + math.pow(yi - yj, 2))
    return int(math.floor(exact_dist + 0.5))


def solve_cvrp(instance_file, time_limit=10):
    data = read_cvrp_instance(instance_file)
    nb_customers = data["nb_customers"]
    nb_trucks = data["nb_trucks"]
    truck_capacity = data["truck_capacity"]

    model = ModelBuilder()
    # Default every list to a full permutation so the kernel can find a feasible
    # starting point for the partition + adjacent-index lambda construction.
    customers_sequences = [
        model.list(nb_customers, default=tuple(range(nb_customers)))
        for _ in range(nb_trucks)
    ]
    model.constraint(model.partition(customers_sequences), name="partition")

    demands = model.array(data["demands"])
    dist_matrix = model.array(data["distance_matrix"])
    dist_depot = model.array(data["distance_depot"])

    # trucks_used = (count(sequence) > 0)
    trucks_used = [(model.count(customers_sequences[k]) > 0) for k in range(nb_trucks)]

    dist_routes = []
    for k in range(nb_trucks):
        sequence = customers_sequences[k]
        c = model.count(sequence)

        # Truck capacity
        demand_lambda = model.lambda_function(lambda j: demands[j])
        route_quantity = model.sum(sequence, demand_lambda)
        model.constraint(route_quantity <= truck_capacity, name=f"cap_{k}")

        # Per-truck distance: sum of inter-customer distances plus depot round-trip
        dist_lambda = model.lambda_function(
            lambda i: dist_matrix[sequence[i - 1], sequence[i]]
        )
        dist_routes.append(
            model.sum(model.range(1, c), dist_lambda)
            + model.iif(c > 0, dist_depot[sequence[0]] + dist_depot[sequence[c - 1]], 0)
        )

    nb_trucks_used = model.sum(*trucks_used)
    total_distance = model.sum(*dist_routes)

    # Combine lexicographic objectives into a single weighted objective.
    # TRUCK_PENALTY exceeds any feasible total_distance value so that minimizing
    # nb_trucks_used strictly dominates minimizing total_distance.
    TRUCK_PENALTY = 10_000_000
    combined = nb_trucks_used * TRUCK_PENALTY + total_distance
    model.minimize(combined, name="trucks_then_distance")

    solution = solve(model, time_limit_s=float(time_limit))
    print(f"status: {solution.status.value}")
    print(f"objective (combined): {solution.objective_value}")
    used = sum(1 for k in range(nb_trucks) if solution.variable_values[customers_sequences[k].node_id])
    print(f"trucks_used: {used}/{nb_trucks}")
    total_dist = 0
    for k in range(nb_trucks):
        seq = solution.variable_values[customers_sequences[k].node_id]
        if seq:
            total_dist += sum(data["distance_matrix"][seq[i - 1]][seq[i]] for i in range(1, len(seq)))
            if seq:
                total_dist += data["distance_depot"][seq[0]] + data["distance_depot"][seq[-1]]
            print(f"Truck {k}: customers={[c + 2 for c in seq]} (in data ids)")
    print(f"total_distance: {total_dist}")


if __name__ == '__main__':
    instances = sorted(
        Path('/Users/dongbox/work/opt-agent/examples/examples/hexaly/capacitated_vehicle_routing_problem_cvrp/instances').glob('*.vrp')
    )
    for instance_file in instances[:1]:
        print(f"\n=== {instance_file.name} ===")
        solve_cvrp(instance_file, time_limit=5)
